## Data Preprocessing
## Overview
This notebook performs essential data preprocessing to prepare CIFAR-10 for CNN training:
- Normalization of pixel values
- Train/validation/test split with stratification
- Data shape verification
- Quality assurance checks
## Why This Matters
Well-preprocessed data is critical for:
- Stable neural network training (normalized inputs)
- Unbiased model evaluation (proper data split)
- Robustness evaluation (consistent preprocessing)

In [1]:
# Imports and Setup
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully")

Libraries imported successfully


## Load CIFAR-10 Dataset 
**Objective:** Load raw CIFAR-10 data and verify structure
**Why this approach:**
- TensorFlow provides built-in CIFAR-10 with proper train/test split
- Easy verification of data integrity
- Reproducible dataset for research

In [2]:

## Load dataset
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = cifar10.load_data()
y_train_raw = y_train_raw.flatten()
y_test_raw = y_test_raw.flatten()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']


print("CIFAR-10 Dataset Loaded Successfully")
print("="*50)
print(f"\nTraining set shape: {X_train_raw.shape}")
print(f"Training labels shape: {y_train_raw.shape}")
print(f"Test set shape: {X_test_raw.shape}")
print(f"Test labels shape: {y_test_raw.shape}")
print(f"\nData type (raw): {X_train_raw.dtype}")
print(f"Pixel value range: [{X_train_raw.min()}, {X_train_raw.max()}]")
print(f"Number of classes: {len(class_names)}")
print(f"Classes: {class_names}")

CIFAR-10 Dataset Loaded Successfully

Training set shape: (50000, 32, 32, 3)
Training labels shape: (50000,)
Test set shape: (10000, 32, 32, 3)
Test labels shape: (10000,)

Data type (raw): uint8
Pixel value range: [0, 255]
Number of classes: 10
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']



## Normalization Strategy
**Why Normalize?**
- Neural networks train faster with normalized inputs (gradient stability)
- Prevents saturation in activation functions
- Reduces training time and improves convergence
**Normalization Method:**
- Min-Max normalization to [0, 1]
- Formula: X_normalized = X_raw / 255.0
- Preserves image quality while enabling stable learning




In [3]:
# Normalize to [0, 1]
X_train_norm = X_train_raw.astype('float32') / 255.0
X_test_norm = X_test_raw.astype('float32') / 255.0

print("\n" + "="*25)
print("NORMALIZATION COMPLETE")
print("="*25)
print(f"\nBefore normalization:")
print(f"  Data type: {X_train_raw.dtype}")
print(f"  Value range: [{X_train_raw.min()}, {X_train_raw.max()}]")

print(f"\nAfter normalization:")
print(f"  Data type: {X_train_norm.dtype}")
print(f"  Value range: [{X_train_norm.min():.4f}, {X_train_norm.max():.4f}]")
print(f"  Mean: {X_train_norm.mean():.4f}")
print(f"  Std Dev: {X_train_norm.std():.4f}")

# Verify no NaN values
print(f"\nMissing values: {np.isnan(X_train_norm).sum()}")
print("Normalization successful")


NORMALIZATION COMPLETE

Before normalization:
  Data type: uint8
  Value range: [0, 255]

After normalization:
  Data type: float32
  Value range: [0.0000, 1.0000]
  Mean: 0.4734
  Std Dev: 0.2516

Missing values: 0
Normalization successful


## Data Split Strategy
**Objective:** Create separate datasets for training, validation, and testing
**Split Ratios:**
- **Training (80%):** 40,000 images - Used to train CNN weights
- **Validation (20%):** 10,000 images - Used to tune hyperparameters and prevent overfitting
- **Test (10,000):** Already separate - Final evaluation on unseen data
**Why Stratified Split?**
- Ensures each class is equally represented in train/val/test
- Prevents class imbalance from skewing results
- Critical for fair model evaluation
**Why This Matters for Robustness:**
- Unbiased evaluation requires proper data split
- Validation prevents overfitting to specific adversarial patterns
- Test set remains truly held-out for final robustness assessment

In [4]:
# Stratified split: training → train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_norm, y_train_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_train_raw
)

# Test set stays as is (held-out from the beginning)
X_test = X_test_norm
y_test = y_test_raw

print("\n" + "="*25)
print("DATA SPLIT COMPLETE")
print("="*25)
print(f"\nTraining set: {X_train.shape} (80% of original 50,000)")
print(f"Validation set: {X_val.shape} (20% of original 50,000)")
print(f"Test set: {X_test.shape} (held-out from beginning)")

# Verify stratification
print("\n" + "-"*50)
print("Class Distribution Verification (Stratification Check)")
print("-"*50)

for split_name, y_split in [("Training", y_train), ("Validation", y_val), ("Test", y_test)]:
    unique, counts = np.unique(y_split, return_counts=True)
    print(f"\n{split_name} set:")
    for class_idx, count in zip(unique, counts):
        pct = (count / len(y_split)) * 100
        print(f"  {class_names[class_idx]:12s}: {count:5d} ({pct:5.1f}%)")


DATA SPLIT COMPLETE

Training set: (40000, 32, 32, 3) (80% of original 50,000)
Validation set: (10000, 32, 32, 3) (20% of original 50,000)
Test set: (10000, 32, 32, 3) (held-out from beginning)

--------------------------------------------------
Class Distribution Verification (Stratification Check)
--------------------------------------------------

Training set:
  airplane    :  4000 ( 10.0%)
  automobile  :  4000 ( 10.0%)
  bird        :  4000 ( 10.0%)
  cat         :  4000 ( 10.0%)
  deer        :  4000 ( 10.0%)
  dog         :  4000 ( 10.0%)
  frog        :  4000 ( 10.0%)
  horse       :  4000 ( 10.0%)
  ship        :  4000 ( 10.0%)
  truck       :  4000 ( 10.0%)

Validation set:
  airplane    :  1000 ( 10.0%)
  automobile  :  1000 ( 10.0%)
  bird        :  1000 ( 10.0%)
  cat         :  1000 ( 10.0%)
  deer        :  1000 ( 10.0%)
  dog         :  1000 ( 10.0%)
  frog        :  1000 ( 10.0%)
  horse       :  1000 ( 10.0%)
  ship        :  1000 ( 10.0%)
  truck       :  1000 ( 10

## Data Quality Verification
**Checks Performed:**
1. Missing values (NaN/Inf)
2. Class balance across splits
3. Data type consistency
4. Value range validation
5. Shape consistency
**Why QA is Critical:**
- Catches preprocessing errors before model training
- Prevents silent failures during training
- Ensures reproducibility




In [5]:
print("\n" + "="*25)
print("DATA QUALITY ASSURANCE")
print("="*25)

# Check for missing values
print("\n1. Missing Values Check:")
print(f"   Training: {np.isnan(X_train).sum()} NaN, {np.isinf(X_train).sum()} Inf")
print(f"   Validation: {np.isnan(X_val).sum()} NaN, {np.isinf(X_val).sum()} Inf")
print(f"   Test: {np.isnan(X_test).sum()} NaN, {np.isinf(X_test).sum()} Inf")
print("No missing values detected")

# Check class balance
print("\n2. Class Balance Check:")
print(f"   Training: {len(np.unique(y_train))} classes")
print(f"   Validation: {len(np.unique(y_val))} classes")
print(f"   Test: {len(np.unique(y_test))} classes")
print("   All classes represented")

# Check data types
print("\n3. Data Type Check:")
print(f"   X_train: {X_train.dtype}")
print(f"   y_train: {y_train.dtype}")
print(f"   X_test: {X_test.dtype}")

# Check value ranges
print("\n4. Value Range Check:")
print(f"   X_train: [{X_train.min():.4f}, {X_train.max():.4f}]")
print(f"   X_val: [{X_val.min():.4f}, {X_val.max():.4f}]")
print(f"   X_test: [{X_test.min():.4f}, {X_test.max():.4f}]")
print("   All within [0.0, 1.0] range")

# Check shapes
print("\n5. Shape Consistency:")
assert X_train.shape[1:] == (32, 32, 3), "Image shape mismatch"
assert X_val.shape[1:] == (32, 32, 3), "Image shape mismatch"
assert X_test.shape[1:] == (32, 32, 3), "Image shape mismatch"
print("   All images are 32×32×3 (RGB)")

print("\n" + "="*40)
print("ALL QUALITY CHECKS PASSED")



DATA QUALITY ASSURANCE

1. Missing Values Check:
   Training: 0 NaN, 0 Inf
   Validation: 0 NaN, 0 Inf
   Test: 0 NaN, 0 Inf
No missing values detected

2. Class Balance Check:
   Training: 10 classes
   Validation: 10 classes
   Test: 10 classes
   All classes represented

3. Data Type Check:
   X_train: float32
   y_train: uint8
   X_test: float32

4. Value Range Check:
   X_train: [0.0000, 1.0000]
   X_val: [0.0000, 1.0000]
   X_test: [0.0000, 1.0000]
   All within [0.0, 1.0] range

5. Shape Consistency:
   All images are 32×32×3 (RGB)

ALL QUALITY CHECKS PASSED


## Save Preprocessed Data for Next Notebook

Save processed datasets to avoid reprocessing

In [6]:
import pickle
import os

# Create data directory
os.makedirs('data/processed', exist_ok=True)

# Save all splits
preprocessed_data = {
    'X_train': X_train,
    'y_train': y_train,
    'X_val': X_val,
    'y_val': y_val,
    'X_test': X_test,
    'y_test': y_test,
    'class_names': class_names
}

with open('data/processed/cifar10_preprocessed.pkl', 'wb') as f:
    pickle.dump(preprocessed_data, f)

print("Preprocessed data saved to 'data/processed/cifar10_preprocessed.pkl'")


Preprocessed data saved to 'data/processed/cifar10_preprocessed.pkl'
